In [1]:
import pandas as pd

# Load the dataset
df = pd.read_csv('/content/mecaniqa_dataset.csv')

# Display the first 5 rows of the DataFrame
display(df.head())

,Data,Trocas_Oleo,Manutencao_Motor
0,2024-01-01,11.0,2.0
1,2024-01-02,9.0,4.0
2,2024-01-03,12.0,1.0
3,2024-01-04,15.0,8.0
4,2024-01-05,25.0,10.0


In [2]:
# Converter a coluna 'Data' para o formato datetime
df['Data'] = pd.to_datetime(df['Data'])

# Ordenar o DataFrame pela coluna 'Data' para garantir a cronologia
df = df.sort_values(by='Data').reset_index(drop=True)

# Exibir as primeiras linhas após a conversão e ordenação
display(df.head())

# Verificar os tipos de dados para confirmar a conversão
display(df.info())

,Data,Trocas_Oleo,Manutencao_Motor
0,2024-01-01,11.0,2.0
1,2024-01-02,9.0,4.0
2,2024-01-03,12.0,1.0
3,2024-01-04,15.0,8.0
4,2024-01-05,25.0,10.0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 731 entries, 0 to 730
Data columns (total 3 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Data              731 non-null    datetime64[ns]
 1   Trocas_Oleo       725 non-null    float64       
 2   Manutencao_Motor  728 non-null    float64       
dtypes: datetime64[ns](1), float64(2)
memory usage: 17.3 KB


None

In [3]:
# Preencher valores ausentes usando ffill (forward fill) e bfill (backward fill)
df['Trocas_Oleo'] = df['Trocas_Oleo'].ffill().bfill()
df['Manutencao_Motor'] = df['Manutencao_Motor'].ffill().bfill()

# Verificar se ainda há valores ausentes
print("Valores ausentes após imputação:")
print(df.isnull().sum())

# Definir as features (X) e os alvos (y) para ambos os modelos
X = df[['Data']]
y_oleo = df['Trocas_Oleo']
y_motor = df['Manutencao_Motor']

# Definir um ponto de corte temporal para a divisão de treino e teste
# Vamos usar 80% dos dados para treino e 20% para teste, mantendo a cronologia
split_point = int(len(df) * 0.8)

# Dividir os dados de forma cronológica
X_train = X.iloc[:split_point]
X_test = X.iloc[split_point:]
y_oleo_train = y_oleo.iloc[:split_point]
y_oleo_test = y_oleo.iloc[split_point:]
y_motor_train = y_motor.iloc[:split_point]
y_motor_test = y_motor.iloc[split_point:]

print(f"\nTamanho do conjunto de treino: {len(X_train)} amostras")
print(f"Tamanho do conjunto de teste: {len(X_test)} amostras")
print(f"Período de treino: {X_train['Data'].min().date()} a {X_train['Data'].max().date()}")
print(f"Período de teste: {X_test['Data'].min().date()} a {X_test['Data'].max().date()}")


Valores ausentes após imputação:
Data                0
Trocas_Oleo         0
Manutencao_Motor    0
dtype: int64

Tamanho do conjunto de treino: 584 amostras
Tamanho do conjunto de teste: 147 amostras
Período de treino: 2024-01-01 a 2025-08-06
Período de teste: 2025-08-07 a 2025-12-31


In [4]:
# Função para extrair características temporais
def extract_time_features(df):
    df['year'] = df['Data'].dt.year
    df['month'] = df['Data'].dt.month
    df['day'] = df['Data'].dt.day
    df['dayofweek'] = df['Data'].dt.dayofweek
    df['dayofyear'] = df['Data'].dt.dayofyear
    df['weekofyear'] = df['Data'].dt.isocalendar().week.astype(int)
    # Adicionar um recurso de "dia do ano" normalizado entre 0 e 1, útil para sazonalidade
    df['normalized_dayofyear'] = df['dayofyear'] / 365.25
    return df.drop('Data', axis=1)

# Aplicar a extração de features aos conjuntos de treino e teste
X_train_processed = extract_time_features(X_train.copy())
X_test_processed = extract_time_features(X_test.copy())

# Exibir as primeiras linhas das features processadas para verificar
print("X_train_processed.head():")
display(X_train_processed.head())
print("\nX_test_processed.head():")
display(X_test_processed.head())

# Importar modelos e métricas
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

# Função para calcular MAPE
def mean_absolute_percentage_error(y_true, y_pred):
    # Evitar divisão por zero
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

# Instanciar modelos
rf_regressor = RandomForestRegressor(random_state=42)
gb_regressor = GradientBoostingRegressor(random_state=42)

models = {
    "RandomForest": rf_regressor,
    "GradientBoosting": gb_regressor
}

targets = {
    "Trocas_Oleo": {'train': y_oleo_train, 'test': y_oleo_test},
    "Manutencao_Motor": {'train': y_motor_train, 'test': y_motor_test}
}

results = {}

# Treinar e avaliar modelos para cada alvo
for target_name, y_data in targets.items():
    print(f"\n----- Treinando e Avaliando para {target_name} -----")
    results[target_name] = {}
    for model_name, model in models.items():
        print(f"Treinando {model_name}...")
        model.fit(X_train_processed, y_data['train'])
        predictions = model.predict(X_test_processed)

        rmse = np.sqrt(mean_squared_error(y_data['test'], predictions))
        mape = mean_absolute_percentage_error(y_data['test'], predictions)

        results[target_name][model_name] = {
            'RMSE': rmse,
            'MAPE': mape
        }
        print(f"{model_name} - RMSE: {rmse:.2f}, MAPE: {mape:.2f}%")

print("\nResultados Finais:")
for target_name, model_results in results.items():
    print(f"Target: {target_name}")
    for model_name, metrics in model_results.items():
        print(f"  {model_name}: RMSE={metrics['RMSE']:.2f}, MAPE={metrics['MAPE']:.2f}%")

X_train_processed.head():


,year,month,day,dayofweek,dayofyear,weekofyear,normalized_dayofyear
0,2024,1,1,0,1,1,0.002738
1,2024,1,2,1,2,1,0.005476
2,2024,1,3,2,3,1,0.008214
3,2024,1,4,3,4,1,0.010951
4,2024,1,5,4,5,1,0.013689



X_test_processed.head():


,year,month,day,dayofweek,dayofyear,weekofyear,normalized_dayofyear
584,2025,8,7,3,219,32,0.599589
585,2025,8,8,4,220,32,0.602327
586,2025,8,9,5,221,32,0.605065
587,2025,8,10,6,222,32,0.607803
588,2025,8,11,0,223,33,0.610541



----- Treinando e Avaliando para Trocas_Oleo -----
Treinando RandomForest...
RandomForest - RMSE: 5.52, MAPE: 22.38%
Treinando GradientBoosting...
GradientBoosting - RMSE: 4.78, MAPE: 17.81%

----- Treinando e Avaliando para Manutencao_Motor -----
Treinando RandomForest...
RandomForest - RMSE: 3.02, MAPE: 36.02%
Treinando GradientBoosting...
GradientBoosting - RMSE: 2.64, MAPE: 31.83%

Resultados Finais:
Target: Trocas_Oleo
  RandomForest: RMSE=5.52, MAPE=22.38%
  GradientBoosting: RMSE=4.78, MAPE=17.81%
Target: Manutencao_Motor
  RandomForest: RMSE=3.02, MAPE=36.02%
  GradientBoosting: RMSE=2.64, MAPE=31.83%
